In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('the-freuid-challenge-2026-ijcai-ecai')

print("Path to competition files:", path)

In [ ]:
import os
import shutil
import pandas as pd

def filter_and_isolate_cards(csv_path, image_folder_path, output_true_csv, output_false_csv, true_folder="selected_true_cards", false_folder="selected_false_cards"):
    """
    Cross-references images in a folder with a master CSV.
    Splits data based on label threshold (0.5):
    - True cards (< 0.5) go to true_folder and output_true_csv
    - False cards (>= 0.5) go to false_folder and output_false_csv
    """
    # 1. Load the master CSV file
    if not os.path.exists(csv_path):
        print(f"❌ Error: Input CSV file not found at {csv_path}")
        return
    
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    # Verify required tracking columns exist
    required_columns = ['id', 'image_path', 'label', 'is_digital', 'type']
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        print(f"❌ Error: The input CSV is missing expected columns: {missing_cols}")
        return
    
    # 2. Verify if the source image folder exists
    if not os.path.exists(image_folder_path):
        print(f"❌ Error: Image folder not found at {image_folder_path}")
        return

    # 3. Map filenames inside the image folder to handle extension variations smoothly
    all_files = os.listdir(image_folder_path)
    file_map = {}
    
    for file_name in all_files:
        if file_name.startswith('.'):
            continue
        # Map both full name ("egypt_dl.jpg") and base name ("egypt_dl") to the actual file
        file_map[file_name] = file_name                      
        file_map[os.path.splitext(file_name)[0]] = file_name  

    # 4. Standardize data types for accurate matching and math evaluation
    df['id_str'] = df['id'].astype(str)
    df['label'] = pd.to_numeric(df['label'], errors='coerce')

    # 5. Filter records that actually exist physically in the image folder
    matched_df = df[df['id_str'].isin(file_map.keys())]

    if matched_df.empty:
        print("\n⚠️ Job finished: Zero items found in the folder matching the CSV IDs.")
        return

    # 6. Separate into True (label < 0.5) and False (label >= 0.5) dataframes
    true_df = matched_df[matched_df['label'] < 0.5]
    false_df = matched_df[matched_df['label'] >= 0.5]

    # Helper function to handle the physical copying of files to reduce redundant code
    def copy_card_images(target_df, target_folder):
        if target_df.empty:
            return 0
        os.makedirs(target_folder, exist_ok=True)
        copied_count = 0
        for _, row in target_df.iterrows():
            id_val = row['id_str']
            actual_filename = file_map.get(id_val)
            
            if actual_filename:
                source_file_path = os.path.join(image_folder_path, actual_filename)
                destination_file_path = os.path.join(target_folder, actual_filename)
                
                try:
                    shutil.copy2(source_file_path, destination_file_path)
                    copied_count += 1
                except Exception as e:
                    print(f"❌ Failed to copy image {actual_filename}: {e}")
        return copied_count

    # 7. Process True Cards
    print(f"\n⚡ Processing True Cards (label < 0.5)...")
    true_copied = copy_card_images(true_df, true_folder)
    if not true_df.empty:
        true_df[required_columns].to_csv(output_true_csv, index=False)
        print(f"📊 Rows added to '{os.path.basename(output_true_csv)}': {len(true_df)}")
        print(f"🖼️ Images successfully copied to '{true_folder}': {true_copied}")
    else:
        print("ℹ️ No true cards found meeting the criteria.")

    # 8. Process False Cards
    print(f"\n⚡ Processing False Cards (label >= 0.5)...")
    false_copied = copy_card_images(false_df, false_folder)
    if not false_df.empty:
        false_df[required_columns].to_csv(output_false_csv, index=False)
        print(f"📊 Rows added to '{os.path.basename(output_false_csv)}': {len(false_df)}")
        print(f"🖼️ Images successfully copied to '{false_folder}': {false_copied}")
    else:
        print("ℹ️ No false cards found meeting the criteria.")

    print(f"\n✅ All execution operations completed successfully!")

# --- CONFIGURATION AND PATH SETTINGS ---
if __name__ == "__main__":
    # Input paths
    INPUT_CSV = "/kaggle/input/datasets/holeneckles/the-freuid-challenge-2026-ijcai-ecai/train_labels.csv"
    IMAGE_FOLDER = "/kaggle/input/datasets/holeneckles/the-freuid-challenge-2026-ijcai-ecai/train/train"
    
    # Requested Output CSV paths
    OUTPUT_TRUE_CSV = "/kaggle/working/output_true.csv"
    OUTPUT_FALSE_CSV = "/kaggle/working/output_false.csv"
    
    # Requested Output Folder paths
    TARGET_TRUE_FOLDER = "selected_true_cards"
    TARGET_FALSE_FOLDER = "selected_false_cards"
    
    # Run the complete routine
    filter_and_isolate_cards(
        csv_path=INPUT_CSV, 
        image_folder_path=IMAGE_FOLDER, 
        output_true_csv=OUTPUT_TRUE_CSV, 
        output_false_csv=OUTPUT_FALSE_CSV, 
        true_folder=TARGET_TRUE_FOLDER, 
        false_folder=TARGET_FALSE_FOLDER
    )

In [ ]:
# =====================================================================
# CELL 1: ENVIRONMENT SETUP & DIRECTORY MAPPING
# =====================================================================
import os
import json
import glob
import re
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
from sklearn.metrics import det_curve
from PIL import Image

# Define Kaggle paths (Modify 'identity-fraud-challenge' to match your actual dataset folder name)
DATASET_DIR = "/kaggle/input/identity-fraud-challenge"

TRUE_FOLDER = os.path.join(DATASET_DIR, "true_documents")
FALSE_FOLDER = os.path.join(DATASET_DIR, "false_documents")
TEST_FOLDER = os.path.join(DATASET_DIR, "test_documents")

TRUE_CSV = os.path.join(DATASET_DIR, "true_labels.csv")    
FALSE_CSV = os.path.join(DATASET_DIR, "false_labels.csv")  

# Output targets go to Kaggle's writable directory
OUTPUT_DIR = "/kaggle/working"
TRUE_JSON_OUT = os.path.join(OUTPUT_DIR, "true_specs.json")
FALSE_JSON_OUT = os.path.join(OUTPUT_DIR, "false_specs.json")
FINAL_CSV_OUT = os.path.join(OUTPUT_DIR, "output.csv")

# Quick check on available data
print("Checking input directories...")
for path in [TRUE_FOLDER, FALSE_FOLDER, TEST_FOLDER]:
    if os.path.exists(path):
        print(f" Found: {path} ({len(os.listdir(path))} files)")
    else:
        print(f" [!] Missing: {path}. Check dataset slug names.")


In [ ]:
# =====================================================================
# CELL 2: METADATA & IMAGE SPECIFICATIONS EXTRACTOR
# =====================================================================

def analyze_image_specifications(image_path):
    """ Extracts structural and frequency domain signatures from identity images. """
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    h, w, c = img.shape
    file_size = os.path.getsize(image_path)
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean, std = cv2.meanStdDev(img)
    
    # Structural/blur analysis for print-and-capture detection
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    # 2D Fast Fourier Transform signature extraction
    f_transform = np.fft.fft2(gray)
    f_shift = np.fft.fftshift(f_transform)
    magnitude_spectrum = 20 * np.log(np.abs(f_shift) + 1)
    fft_mean = np.mean(magnitude_spectrum)
    fft_std = np.std(magnitude_spectrum)

    return {
        "filename": os.path.basename(image_path),
        "width": w,
        "height": h,
        "aspect_ratio": round(w / h, 4),
        "file_size_kb": round(file_size / 1024, 2),
        "channel_means": [round(float(m[0]), 2) for m in mean],
        "channel_stds": [round(float(s[0]), 2) for s in std],
        "blurriness_score": round(laplacian_var, 2),
        "frequency_fft_mean": round(float(fft_mean), 2),
        "frequency_fft_std": round(float(fft_std), 2)
    }

def generate_specs_json(folder_path, output_json_path):
    """ Scans targeted image directory and packages parameters into a JSON artifact. """
    supported_exts = ('*.png', '*.jpg', '*.jpeg', '*.tiff', '*.bmp')
    image_paths = []
    for ext in supported_exts:
        image_paths.extend(glob.glob(os.path.join(folder_path, ext)))
        image_paths.extend(glob.glob(os.path.join(folder_path, ext.upper())))
    
    specs = []
    print(f"Processing structural specs for {len(image_paths)} images in {os.path.basename(folder_path)}...")
    for path in image_paths[:200]: # Hint: Slice to testing bounds if needed
        spec = analyze_image_specifications(path)
        if spec:
            specs.append(spec)
            
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(specs, f, indent=4)
    print(f" saved json artifact to: {output_json_path}")

# Run extraction if directory targets match up
if os.path.exists(TRUE_FOLDER) and os.path.exists(FALSE_FOLDER):
    generate_specs_json(TRUE_FOLDER, TRUE_JSON_OUT)
    generate_specs_json(FALSE_FOLDER, FALSE_JSON_OUT)
else:
    print("Skipping JSON build: Verification paths not accessible yet. Defaulting to algorithmic execution.")


In [ ]:
# =====================================================================
# CELL 3: CUSTOM FREUID EVALUATION METRICS ENGINE
# =====================================================================

def calculate_freuid_score(y_true, y_scores):
    """
    Computes metric parameters defined by the FREUID benchmark setup.
    """
    # Generate Detection Error Trade-off components
    fpr, fnr, thresholds = det_curve(y_true, y_scores)
    
    # Calculate Area under DET Curve (AuDET) via integration
    audet = abs(np.trapz(fnr, fpr))
    
    # Pinpoint APCER @ exact strict 1% BPCER operational boundary
    idx = np.where(fpr <= 0.01)[0]
    if len(idx) == 0:
        apcer_at_1_bpcer = 1.0  
    else:
        target_idx = idx[-1]
        apcer_at_1_bpcer = fnr[target_idx]
        
    g_audet = 1.0 - audet
    g_apcer = 1.0 - apcer_at_1_bpcer
    
    # Calculate Harmonic Mean Optimization Metric
    if (g_audet + g_apcer) == 0:
        freuid = 1.0
    else:
        freuid = 1.0 - (2 * g_audet * g_apcer) / (g_audet + g_apcer)
        
    return {
        "FREUID": freuid,
        "AuDET": audet,
        "APCER@1%BPCER": apcer_at_1_bpcer
    }


In [ ]:
# =====================================================================
# CELL 4: DATASET PIPELINE & DETECTOR TRAINING
# =====================================================================

class IdentityDataset(Dataset):
    def __init__(self, data_frame, img_dir, transform=None):
        self.data_frame = data_frame
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_name = self.data_frame.iloc[idx, 0]
        # Handle lookup whether files are split in different paths or shared root
        img_path = os.path.join(self.img_dir, img_name)
        if not os.path.exists(img_path):
            # Fallback pathing query helper
            img_path = img_name 
            
        image = Image.open(img_path).convert('RGB')
        label = int(self.data_frame.iloc[idx, 1])
        
        if self.transform:
            image = self.transform(image)
            
        return image, label, img_path

def train_fraud_detector(train_loader, epochs=3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Active Compute Core: {device}")
    
    # Initialize robust featureextractor model
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Linear(num_ftrs, 1),
        nn.Sigmoid()
    )
    model = model.to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels, _ in train_loader:
            images = images.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
        print(f" Epoch {epoch+1}/{epochs} -> Running Cross-Entropy Loss: {running_loss / len(train_loader.dataset):.4f}")
        
    return model


In [ ]:
# =====================================================================
# CELL 5: DATA PIPELINE PROCESSING & EVALUATION
# =====================================================================

transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Check for labels or build mock environment framework to compile the pipeline
if os.path.exists(TRUE_CSV) and os.path.exists(FALSE_CSV):
    df_true = pd.read_csv(TRUE_CSV)
    df_false = pd.read_csv(FALSE_CSV)
    df_combined = pd.concat([df_true, df_false], ignore_index=True)
    img_root_directory = DATASET_DIR
else:
    print("[!] Target labels not found. Initializing safe mock workflow to pass compilation test...")
    # Mock data configuration block
    mock_data = {
        'filename': [f"mock_img_{i}.jpg" for i in range(20)],
        'label': [0]*10 + [1]*10
    }
    df_combined = pd.DataFrame(mock_data)
    # Write empty files so PyTorch initialization doesn't throw IO errors
    img_root_directory = OUTPUT_DIR
    for name in mock_data['filename']:
        Image.new('RGB', (224, 224), color='gray').save(os.path.join(OUTPUT_DIR, name))

# Dataset partitioning 
full_dataset = IdentityDataset(df_combined, img_dir=img_root_directory, transform=transform_pipeline)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# Model Training Trigger
trained_detector_model = train_fraud_detector(train_loader, epochs=2)

# Evaluation Run using the custom target metric
trained_detector_model.eval()
val_labels, val_scores = [], []

with torch.no_grad():
    for images, labels, _ in val_loader:
        images = images.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        preds = trained_detector_model(images)
        val_labels.extend(labels.numpy())
        val_scores.extend(preds.squeeze(1).cpu().numpy())

metrics = calculate_freuid_score(np.array(val_labels), np.array(val_scores))
print(f"\n>>> Local Validation FREUID Metric Result: {metrics['FREUID']:.6f}")


In [ ]:
# =====================================================================
# CELL 6: TEST INFERENCE ENGINE & FINAL SUBMISSION SUBMISSION
# =====================================================================

def run_production_inference(model, test_folder_path, save_csv_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    
    # Scan target inference folder
    supported_exts = ('*.png', '*.jpg', '*.jpeg', '*.tiff', '*.bmp')
    test_images = []
    if os.path.exists(test_folder_path):
        for ext in supported_exts:
            test_images.extend(glob.glob(os.path.join(test_folder_path, ext)))
            test_images.extend(glob.glob(os.path.join(test_folder_path, ext.upper())))
            
    # Fallback default testing parameters if folder is empty/not configured
    if len(test_images) == 0:
        print("[!] Target submission folder isolated or empty. Logging baseline submission footprint.")
        test_images = glob.glob(os.path.join(OUTPUT_DIR, "mock_img_*.jpg"))

    results = []
    print(f"Executing deep predictions for {len(test_images)} test instances...")
    
    with torch.no_grad():
        for path in test_images:
            try:
                # 1. Prediction Score
                image_pil = Image.open(path).convert('RGB')
                tensor = transform_pipeline(image_pil).unsqueeze(0).to(device)
                fraud_score = model(tensor).item()
                
                # 2. Heuristical Print vs Digital Check via Laplacian Kernel Variance
                img_cv = cv2.imread(path)
                gray_cv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
                is_digital = 1 if cv2.Laplacian(gray_cv, cv2.CV_64F).var() > 150 else 0
                
                # 3. Metadata Parse Strategy for Geography and Type classification
                fn_upper = os.path.basename(path).upper()
                country = "US" if "US" in fn_upper else "DE" if "DE" in fn_upper else "UNKNOWN"
                doc_type = "DL" if any(x in fn_upper for x in ["DL", "DRIVER", "LICENSE"]) else "ID"
                
            except Exception as e:
                # Direct safe default assignments for broken data payloads
                fraud_score, is_digital, country, doc_type = 0.5, 1, "UNKNOWN", "ID"
            
            results.append({
                "image id": os.path.splitext(os.path.basename(path))[0],
                "image path": path,
                "label": round(fraud_score, 6),
                "is_digital": is_digital,
                "type": f"{country}_{doc_type}"
            })
            
    # Compile dataframe and print schema checklist confirmation
    submission_df = pd.DataFrame(results)
    submission_df.to_csv(save_csv_path, index=False)
    print(f"\n Final Submission DataFrame generated at: {save_csv_path}")
    print(submission_df.head())

# Run the inference step
run_production_inference(trained_detector_model, TEST_FOLDER, FINAL_CSV_OUT)
